# SQL Task: Employee Workforce Data Analysis

This notebook uses SQL to create, manage, and analyze employee workforce data. The analysis focuses on departments, salaries, employee performance, and workforce insights.

## Prompt Used

**Prompt:**

Act as a SQL developer. Create a simple SQL-based employee workforce analysis using sample employee data. Create an employee table containing Employee ID, Employee Name, Department, Job Role, Experience, Salary, Performance Score, and Attrition.

Write SQL queries to create and populate the table and perform basic workforce analysis. The analysis should include viewing employee records, calculating average salary by department, identifying high-performing employees, and analyzing employee attrition.

Use beginner-friendly SQL queries and provide clear outputs and explanations. Keep the task focused on workforce data analysis rather than advanced database concepts.

In [1]:
import sqlite3
import pandas as pd

# Create a fresh SQLite database for this notebook
import os
import sqlite3
import pandas as pd

if os.path.exists("workforce.db"):
    os.remove("workforce.db")

conn = sqlite3.connect("workforce.db")
cursor = conn.cursor()


print("Database created successfully!")

Database created successfully!


## 1. Database Setup

A SQLite database named `workforce.db` was created for the workforce analysis. SQLite allows SQL queries to be executed directly within the Google Colab environment without requiring an external database server.

In [2]:
# Create the employee table
cursor.execute("""
CREATE TABLE IF NOT EXISTS employees (
    Employee_ID INTEGER PRIMARY KEY,
    Employee_Name TEXT NOT NULL,
    Department TEXT NOT NULL,
    Job_Role TEXT NOT NULL,
    Experience_Years INTEGER,
    Salary REAL,
    Performance_Score REAL,
    Attrition TEXT
)
""")

conn.commit()

print("Employee table created successfully!")

Employee table created successfully!


## 2. Insert Employee Data

Sample employee records are inserted into the employee table. The data contains information about employee departments, job roles, experience, salary, performance, and attrition status.

In [3]:
# Insert sample employee records
employees = [
    (101, "Aarav", "IT", "Developer", 2, 45000, 4.2, "No"),
    (102, "Priya", "HR", "HR Executive", 5, 55000, 3.8, "No"),
    (103, "Rahul", "Finance", "Financial Analyst", 8, 70000, 4.5, "No"),
    (104, "Sneha", "Sales", "Sales Executive", 3, 40000, 3.5, "Yes"),
    (105, "Arjun", "IT", "Data Analyst", 6, 65000, 4.6, "No"),
    (106, "Meera", "Marketing", "Marketing Executive", 4, 50000, 4.0, "No"),
    (107, "Karan", "Sales", "Sales Executive", 2, 38000, 3.2, "Yes"),
    (108, "Ananya", "HR", "HR Manager", 7, 68000, 4.3, "No"),
    (109, "Vikram", "Finance", "Accountant", 10, 85000, 4.7, "No"),
    (110, "Riya", "IT", "Senior Developer", 5, 72000, 4.4, "No")
]

cursor.executemany("""
INSERT INTO employees
(Employee_ID, Employee_Name, Department, Job_Role, Experience_Years,
 Salary, Performance_Score, Attrition)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""", employees)

conn.commit()

print("Employee records inserted successfully!")

Employee records inserted successfully!


In [4]:
# Display all employee records
df_sql = pd.read_sql_query("SELECT * FROM employees", conn)

df_sql

,Employee_ID,Employee_Name,Department,Job_Role,Experience_Years,Salary,Performance_Score,Attrition
0,101,Aarav,IT,Developer,2,45000.0,4.2,No
1,102,Priya,HR,HR Executive,5,55000.0,3.8,No
2,103,Rahul,Finance,Financial Analyst,8,70000.0,4.5,No
3,104,Sneha,Sales,Sales Executive,3,40000.0,3.5,Yes
4,105,Arjun,IT,Data Analyst,6,65000.0,4.6,No
5,106,Meera,Marketing,Marketing Executive,4,50000.0,4.0,No
6,107,Karan,Sales,Sales Executive,2,38000.0,3.2,Yes
7,108,Ananya,HR,HR Manager,7,68000.0,4.3,No
8,109,Vikram,Finance,Accountant,10,85000.0,4.7,No
9,110,Riya,IT,Senior Developer,5,72000.0,4.4,No


## 3. Average Salary by Department

This query calculates the average salary for employees in each department. It helps compare salary levels across different areas of the organization.

In [5]:
query = """
SELECT
    Department,
    ROUND(AVG(Salary), 2) AS Average_Salary
FROM employees
GROUP BY Department
ORDER BY Average_Salary DESC;
"""

result = pd.read_sql_query(query, conn)

result

,Department,Average_Salary
0,Finance,77500.00
1,HR,61500.00
2,IT,60666.67
3,Marketing,50000.00
4,Sales,39000.00


### Interpretation

The query shows the average salary for each department. Comparing departmental averages can help identify differences in compensation levels across the workforce.

## 4. High-Performing Employees

This query identifies employees with a performance score of 4.5 or above. It demonstrates how SQL can be used to filter employee records based on workforce performance.

In [6]:
query = """
SELECT
    Employee_ID,
    Employee_Name,
    Department,
    Job_Role,
    Performance_Score
FROM employees
WHERE Performance_Score >= 4.5
ORDER BY Performance_Score DESC;
"""

high_performers = pd.read_sql_query(query, conn)

high_performers

,Employee_ID,Employee_Name,Department,Job_Role,Performance_Score
0,109,Vikram,Finance,Accountant,4.7
1,105,Arjun,IT,Data Analyst,4.6
2,103,Rahul,Finance,Financial Analyst,4.5


### Interpretation

The query identifies employees with relatively high performance scores. Such analysis can help organizations recognize strong performers and understand where high performance exists within the workforce.

## 5. Employee Attrition Analysis

This query calculates the number of employees who have stayed with or left the organization. Attrition analysis can help organizations understand workforce retention patterns.

In [7]:
query = """
SELECT
    Attrition,
    COUNT(*) AS Employee_Count
FROM employees
GROUP BY Attrition;
"""

attrition_result = pd.read_sql_query(query, conn)

attrition_result

,Attrition,Employee_Count
0,No,8
1,Yes,2


### Interpretation

The result shows the number of employees in each attrition category. This provides a simple view of employee retention and turnover within the sample workforce.

## 6. Employees Earning Above Department Average

This query identifies employees whose salary is higher than the average salary of their respective department. It demonstrates the use of a subquery for workforce analysis.

In [8]:
query = """
SELECT
    e.Employee_ID,
    e.Employee_Name,
    e.Department,
    e.Salary,
    ROUND(dept_avg.Average_Salary, 2) AS Department_Average_Salary
FROM employees e
JOIN (
    SELECT
        Department,
        AVG(Salary) AS Average_Salary
    FROM employees
    GROUP BY Department
) dept_avg
ON e.Department = dept_avg.Department
WHERE e.Salary > dept_avg.Average_Salary
ORDER BY e.Salary DESC;
"""

above_average = pd.read_sql_query(query, conn)

above_average

,Employee_ID,Employee_Name,Department,Salary,Department_Average_Salary
0,109,Vikram,Finance,85000.0,77500.00
1,110,Riya,IT,72000.0,60666.67
2,108,Ananya,HR,68000.0,61500.00
3,105,Arjun,IT,65000.0,60666.67
4,104,Sneha,Sales,40000.0,39000.00


### Interpretation

The query compares each employee's salary with the average salary of their department. This provides a more meaningful salary comparison than comparing every employee against the overall workforce average.

## 7. Overall Workforce Summary

The final query provides a basic summary of the employee dataset, including total employees, average salary, and average performance score.

In [9]:
query = """
SELECT
    COUNT(*) AS Total_Employees,
    ROUND(AVG(Salary), 2) AS Average_Salary,
    ROUND(AVG(Performance_Score), 2) AS Average_Performance
FROM employees;
"""

summary = pd.read_sql_query(query, conn)

summary

,Total_Employees,Average_Salary,Average_Performance
0,10,58800.0,4.12


## 8. Conclusion

This SQL task demonstrated how SQL can be used to create and analyze employee workforce data. The queries covered departmental salary analysis, employee performance, attrition, and salary comparisons.

These basic SQL techniques can help organizations retrieve workforce information from databases and support data-driven decision-making.

In [10]:
conn.close()

print("Database connection closed successfully.")

Database connection closed successfully.
